# TPI · Módulo 1 — Contextualización y preprocesamiento
**PSIB 2026Q2 · Grupo 12** — Lautersztein, Reggio, Ballester
*Detección de leucemia linfoblástica aguda (LLA) en imágenes de frotis de sangre periférica*

Este notebook cubre las etapas 1 y 2 del pipeline del anteproyecto:

1. **Contextualización**: características del dataset (clases, formato, resolución), histogramas RGB/HSV por clase, uniformidad de la iluminación (canal V) y variabilidad de la tinción.
2. **Corrección de iluminación**: estimación del campo de iluminación (gaussiano de escala amplia vs. cierre morfológico) y normalización por el fondo, comparadas con CLAHE. Todas actúan **sólo sobre V**, dejando H y S intactos para no alterar el color de la tinción.

La salida es la función `preprocesar()` con el método elegido, que usa el Módulo 2 (segmentación). Las conclusiones, con los resultados sobre el dataset completo, están al final (sección 3).

## 0. Descarga e índice del dataset
*Acute Lymphoblastic Leukemia (ALL) Image Dataset* (Kaggle, Aria et al. 2021). `kagglehub` viene instalado en Colab; si pide credenciales, subir el `kaggle.json` de la cuenta de Kaggle.

Armamos una tabla con una fila por imagen: si es **original** o la **segmentada** por los autores (que usaremos como referencia en el Módulo 2), el **subtipo** (Benign, Early Pre-B, Pre-B, Pro-B) y la **clase binaria** del proyecto (benigna = no patológica; los tres subtipos de LLA = patológica).

In [ ]:
import os, glob
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt
import kagglehub

RUTA = kagglehub.dataset_download('mehradaria/leukemia')     # carpeta local donde quedó el dataset

df = pd.DataFrame({'ruta': sorted(glob.glob(f'{RUTA}/**/*.jp*g', recursive=True))})
df['tipo'] = np.where(df.ruta.str.contains('segment', case=False), 'segmentada', 'original')
df['subtipo'] = df.ruta.map(lambda p: os.path.basename(os.path.dirname(p)))          # nombre de la carpeta
df['clase'] = np.where(df.subtipo.str.contains('benign', case=False), 'no patológica', 'patológica')
df['archivo'] = df.ruta.map(os.path.basename)
df.groupby(['tipo', 'clase', 'subtipo']).size().to_frame('n_imágenes')

In [ ]:
org = df[df.tipo == 'original'].reset_index(drop=True)
# ¿Cada original tiene su segmentación de referencia? (se emparejan por subtipo + nombre de archivo)
seg = df[df.tipo == 'segmentada'].set_index(['subtipo', 'archivo']).ruta
org['ruta_seg'] = org.set_index(['subtipo', 'archivo']).index.map(seg.get)
print(f"Originales: {len(org)} · con segmentación de referencia: {org.ruta_seg.notna().sum()}")
print('Proporción por clase:', org.clase.value_counts(normalize=True).round(3).to_dict())

## 1. Contextualización
### 1.1 Ejemplos por subtipo
En la tinción de May-Grünwald-Giemsa / Wright el **núcleo** del linfocito se ve violeta intenso, el **citoplasma** celeste pálido, los **eritrocitos** rosados y el **fondo** (plasma) casi blanco.

In [ ]:
leer = lambda ruta: cv2.cvtColor(cv2.imread(ruta), cv2.COLOR_BGR2RGB)     # OpenCV lee en BGR: lo pasamos a RGB para mostrar
subtipos = sorted(org.subtipo.unique())
fig, ax = plt.subplots(2, len(subtipos), figsize=(3.5 * len(subtipos), 7))
for j, st in enumerate(subtipos):
    for i, r in enumerate(org[org.subtipo == st].sample(2, random_state=0).itertuples()):
        ax[i, j].imshow(leer(r.ruta)); ax[i, j].axis('off')
    ax[0, j].set_title(st)
plt.tight_layout()

### 1.2 Características globales de cada imagen
En una sola pasada por todas las imágenes originales calculamos:
- **resolución** y **formato**;
- **medias de color** en HSV y CIE Lab (para estudiar la variabilidad de tinción entre muestras);
- **uniformidad de la iluminación**: estimamos el *campo de iluminación* del canal V (el brillo del fondo sin las células) y medimos su coeficiente de variación (CV = σ/μ) y el cociente centro/borde. Un campo uniforme tiene CV ≈ 0 y centro/borde ≈ 1; el viñeteo típico del microscopio da centro/borde > 1;
- **histogramas** RGB y HSV acumulados por clase.

El campo se estima con un **cierre morfológico** (dilatación seguida de erosión) con un elemento estructurante más grande que las células: como las células son más oscuras que el fondo, el cierre las "borra" y deja sólo la iluminación de fondo. Un suavizado gaussiano final evita escalones. Esta misma función se reutiliza en la corrección (sección 2).

In [ ]:
TAM_EE = 1 / 3      # tamaño del elemento estructurante relativo a la imagen: tiene que superar el diámetro de una célula

def estimar_campo(v, metodo='cierre'):
    """Campo de iluminación del canal V (float32).
    'cierre': cierre morfológico con EE elíptico grande -> elimina las células (oscuras) y deja el fondo.
    'gauss' : pasa-bajos gaussiano de escala amplia (σ = 1/4 de la imagen) -> conserva sólo la variación lenta."""
    n = max(v.shape)
    if metodo == 'gauss':
        return cv2.GaussianBlur(v, (0, 0), n / 4)
    ee = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(n * TAM_EE) | 1,) * 2)   # | 1 -> tamaño impar
    return cv2.GaussianBlur(cv2.morphologyEx(v, cv2.MORPH_CLOSE, ee), (0, 0), n / 16)

def centro_borde(campo):
    """Cociente entre el brillo medio del centro y el de un marco de 1/8 de la imagen."""
    b = max(campo.shape) // 8
    marco = np.ones(campo.shape, bool); marco[b:-b, b:-b] = False
    h, w = campo.shape
    return campo[h//2 - b:h//2 + b, w//2 - b:w//2 + b].mean() / campo[marco].mean()

def histogramas(rgb, hsv):
    """Histogramas normalizados de R, G, B, H, S, V -> [6 x 256]."""
    return np.array([np.bincount(c.ravel(), minlength=256)[:256] / c.size
                     for c in np.concatenate([rgb, hsv], axis=2).transpose(2, 0, 1)])

filas, hist = [], {c: 0 for c in org.clase.unique()}
for r in org.itertuples():
    rgb = leer(r.ruta)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)       # OpenCV: H en [0, 180), S y V en [0, 255]
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)       # OpenCV: a* y b* desplazados +128 (128 = neutro)
    campo = estimar_campo(hsv[..., 2].astype(np.float32))
    filas.append(dict(alto=rgb.shape[0], ancho=rgb.shape[1], ext=os.path.splitext(r.ruta)[1].lower(),
                      H=hsv[..., 0].mean(), S=hsv[..., 1].mean(), V=hsv[..., 2].mean(),
                      V_sat=(hsv[..., 2] == 255).mean(),           # fracción de píxeles saturados (fondo "quemado")
                      a=lab[..., 1].mean() - 128, b=lab[..., 2].mean() - 128,
                      cv_campo=campo.std() / campo.mean(), centro_borde=centro_borde(campo)))
    hist[r.clase] = hist[r.clase] + histogramas(rgb, hsv)
desc = pd.concat([org[['subtipo', 'clase']], pd.DataFrame(filas)], axis=1)

print('Resoluciones (alto x ancho):', desc.groupby(['alto', 'ancho']).size().to_dict())
print('Formatos:', desc.ext.value_counts().to_dict())
print(f"Imágenes con iluminación no uniforme (CV del campo > 1 %): {(desc.cv_campo > 0.01).mean():.1%}")
desc.groupby('clase')[['H', 'S', 'V', 'V_sat', 'a', 'b', 'cv_campo', 'centro_borde']].agg(['mean', 'std']).round(3)

### 1.3 Histogramas RGB y HSV por clase
Histograma medio de cada canal para benignas vs. patológicas. Diferencias marcadas (p. ej. en H o S) anticipan qué canales conviene usar para separar el núcleo en la segmentación.

In [ ]:
nombres = ['R', 'G', 'B', 'H', 'S', 'V']
fig, ax = plt.subplots(2, 3, figsize=(14, 6))
for k, a in enumerate(ax.ravel()):
    for clase, h in hist.items():
        a.plot(h[k] / (org.clase == clase).sum(), label=clase)        # promedio de los histogramas de la clase
    a.set(title=nombres[k], xlim=(0, 180 if nombres[k] == 'H' else 255))
ax[0, 0].legend(); plt.tight_layout()

### 1.4 Iluminación y tinción
- **Iluminación**: distribución del CV del campo y del cociente centro/borde. Si la mayoría de las imágenes tiene CV bajo y centro/borde ≈ 1, la corrección de iluminación aporta poco (es lo que pasa acá: ver sección 3).
- **Tinción**: dispersión de las medias de H, S, a\* y b\* por subtipo. Mucha dispersión dentro de un mismo subtipo indica variabilidad de tinción entre muestras, que puede afectar umbrales fijos en la segmentación.

In [ ]:
fig, ax = plt.subplots(1, 6, figsize=(18, 3.5))
for a, var in zip(ax, ['cv_campo', 'centro_borde', 'H', 'S', 'a', 'b']):
    a.boxplot([desc.loc[desc.subtipo == st, var] for st in subtipos])
    a.set_xticks(range(1, len(subtipos) + 1), subtipos, rotation=45)
    a.set_title(var)
plt.tight_layout()

## 2. Corrección de iluminación
Tres alternativas sobre el canal V, comparadas con la imagen sin corregir:

| Método | Qué hace |
|---|---|
| `gauss` | campo = gaussiano de escala amplia; **V / campo** (normalización por el fondo) |
| `cierre` | campo = cierre morfológico; **V / campo** |
| `clahe` | ecualización adaptativa de histograma con límite de contraste (realza contraste local, no estima el campo) |

En la normalización, al dividir por el campo se cancela la iluminación no uniforme, y al multiplicar por su media se conserva el brillo global.

In [ ]:
METODOS = ['ninguno', 'gauss', 'cierre', 'clahe']
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocesar(rgb, metodo='cierre'):
    """Corrige la iluminación sólo en V (H y S intactos). Devuelve la imagen RGB corregida."""
    h, s, v = cv2.split(cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV))
    if metodo == 'clahe':
        v = clahe.apply(v)
    elif metodo != 'ninguno':
        campo = np.maximum(estimar_campo(v.astype(np.float32), metodo), 1)     # evita dividir por 0
        v = np.clip(v / campo * campo.mean(), 0, 255).astype(np.uint8)
    return cv2.cvtColor(cv2.merge([h, s, v]), cv2.COLOR_HSV2RGB)

### 2.1 Comparación visual y perfiles de iluminación
Para una imagen de ejemplo: resultado de cada método y perfil horizontal (fila central) del campo de iluminación estimado **después** de corregir. Si la corrección funciona, el perfil queda plano.

In [ ]:
ej = leer(org.ruta[desc.cv_campo.idxmax()])          # la imagen con iluminación menos uniforme del dataset
fig, ax = plt.subplots(1, len(METODOS) + 1, figsize=(20, 4))
for a, m in zip(ax, METODOS):
    corr = preprocesar(ej, m)
    campo = estimar_campo(cv2.cvtColor(corr, cv2.COLOR_RGB2HSV)[..., 2].astype(np.float32))
    a.imshow(corr); a.set_title(m); a.axis('off')
    ax[-1].plot(campo[campo.shape[0] // 2], label=m)
ax[-1].set(title='Perfil del campo (fila central)', xlabel='Columna [px]', ylabel='V'); ax[-1].legend()
plt.tight_layout()

### 2.2 Comparación cuantitativa
Medimos cada método sobre dos grupos: una **muestra aleatoria** estratificada (150 benignas + 150 patológicas) y **todas las imágenes no uniformes** (CV del campo > 1 %), que son donde la corrección debería notarse.
- **cv_fondo**: dispersión del brillo en el fondo (σ/μ de V). Cuanto menor, más uniforme quedó la iluminación.
- **contraste**: contraste núcleo–fondo $(g_f - g_n)/(g_f + g_n)$ en **escala de grises**. Conviene que no baje.
- **detalle_nuclear**: σ/μ del gris dentro del núcleo (variación de la cromatina). Parecido a `ninguno` → se **preservan** los detalles; mucho mayor → se amplifica ruido.

El contraste se mide en gris y no en V porque **en este dataset el núcleo no es oscuro en V**: V = máx(R, G, B) y el violeta del núcleo tiene el azul alto, así que su V (≈ 240) es casi igual al del fondo. El núcleo se separa por **saturación y tono** (S y H), y en luminancia.

Las regiones se aproximan **en la imagen original**, para comparar todos los métodos sobre los mismos píxeles: fondo = 30 % de píxeles menos saturados (plasma) y núcleo = 5 % más saturados (violeta intenso). Es una aproximación rápida; la segmentación real es el Módulo 2.

In [ ]:
def regiones(rgb):
    """Máscaras aproximadas (fondo, núcleo) a partir de la saturación de la imagen ORIGINAL."""
    s = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[..., 1]
    return s < np.percentile(s, 30), s > np.percentile(s, 95)

def metricas(rgb, fondo, nucleo):
    """Métricas de calidad de una imagen (ya corregida) evaluadas sobre las máscaras de la original."""
    v = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[..., 2].astype(float)
    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(float)
    gf, gn = g[fondo].mean(), g[nucleo].mean()
    return dict(cv_fondo=v[fondo].std() / v[fondo].mean(), contraste=(gf - gn) / (gf + gn), detalle_nuclear=g[nucleo].std() / gn)

grupos = {'aleatoria': org.groupby('clase').sample(150, random_state=0).index,   # 150 benignas + 150 patológicas
          'no uniforme': desc.index[desc.cv_campo > 0.01]}
res = []
for grupo, idx in grupos.items():
    for ruta in org.ruta[idx]:
        rgb = leer(ruta)
        fondo, nucleo = regiones(rgb)
        res += [dict(grupo=grupo, metodo=m, **metricas(preprocesar(rgb, m), fondo, nucleo)) for m in METODOS]
res = pd.DataFrame(res)
res.groupby(['grupo', 'metodo'], sort=False)[['cv_fondo', 'contraste', 'detalle_nuclear']].mean().round(3)

## 3. Conclusiones del Módulo 1
*(Corrida sobre las 3.256 imágenes originales.)*

**Dataset**
- 3.256 imágenes JPG de 224 × 224 px, todas con su segmentación de referencia. Son campos con muchas células (no recortes de una sola célula).
- Desbalance marcado: 15,5 % benignas (504) vs. 84,5 % patológicas (2.752). Va a haber que tenerlo en cuenta en la división ajuste/prueba y al reportar métricas.
- Muchas imágenes tienen una **marca de agua o barra de escala en la esquina inferior derecha**. En la segmentación hay que descartar los objetos de esa zona.

**Iluminación**
- Es casi uniforme: el CV del campo supera el 1 % sólo en el **3,3 %** de las imágenes (proporcionalmente más en las benignas: 7,5 %).
- El fondo está **saturado**: ~26 % de los píxeles tienen V = 255.
- En las imágenes no uniformes, el **cierre morfológico** y el gaussiano bajan la dispersión del fondo ~15 % (0,048 → 0,041) sin perder contraste ni detalle nuclear. En la peor imagen el cierre deja el perfil más plano.
- **CLAHE se descarta**: casi duplica la dispersión del fondo y aumenta ~35 % la variación dentro del núcleo, o sea, amplifica ruido.
- **Decisión: se aplica el cierre a todas las imágenes.** En las uniformes no cambia nada (tabla: grupo aleatorio) y corrige las que lo necesitan.

**Color y tinción**
- El núcleo **no es oscuro en V**: se separa por S y H (y b\* en Lab). Esto respalda segmentar con umbrales en H–S y *k*-means en a\*b\*, como propone el anteproyecto, y no con V.
- La tinción **varía mucho entre subtipos**: Pre-B es mucho más saturado (S ≈ 59) y Pro-B mucho más pálido (S ≈ 18) que el resto. Es esperable si cada subtipo proviene de pacientes y tandas de tinción distintos. Consecuencia: los descriptores de **color global** podrían separar las clases por la tinción y no por la morfología. Conviene priorizar forma y textura, y/o normalizar el color antes de extraer descriptores de color.

In [ ]:
METODO_ELEGIDO = 'cierre'    # ver conclusiones: corrige el 3,3 % no uniforme sin afectar al resto
desc.to_csv('modulo1_descriptores_globales.csv', index=False)